# PARFLM on TinyStories — P10 ladder (target: val PPL ≤ 20) on A100 / H100

## Where the wall is

On TinyStories at the v3 paper's scale-up cell shape (`d=256, L=8, T=512, B=16, v_hidden=1024, 8000 steps`), the relevant bookend numbers are:

| Reference | Val PPL | Source |
| --- | --- | --- |
| **SPLM em_ln single-ξ (leak-fixed)** | **33.55** | `docs/Restructuring_paper_v3_after_causal_leak_bug.md` line 22 — the wall the old PARF-augmented SPLM was failing to surpass |
| MatchedGPT param-matched attention | 7.81 | upper bound to chase (8000 steps) |

**P10 success criterion:** the new PARFLM (P5 sparsity + P7 competitive Φ + P8 composite cell) lands at **val PPL ≤ 20** on this same TinyStories cell, with **≤ 25 PPL** as the secondary acceptable result and **anything beating SPLM em_ln 33.55** as the minimum non-trivial outcome.

## P10 ladder

Four cells, each one configurable via the `CELL` switch in the setup cell below. Run the notebook end-to-end four times, swapping `CELL` each pass. Each P10 cell has its own output dir under `RESULTS_ROOT/{cell}/seed{seed}/`, where `RESULTS_ROOT` is `/content/drive/MyDrive/semsimula_parflm/p10_tinystories/` on Colab (auto-bootstrapped, see §0) and `parf/results/p10_tinystories/` when run locally.

| Cell | Composition | Predicted val PPL | What it tests |
| --- | --- | --- | --- |
| `P10a` | `--v-phi-kind structural` + `--top-k 4` | 33–35 (actual: **32.60**) | Anchors the curve; replicates the SPLM em_ln ceiling on TinyStories with the P5 sparse winner. |
| `P10b` | P10a + `--ln-before-distance --per-layer-v-phi-scale --theta-activation softsign --theta-form bilinear` | 28–32 | Isolates the P8 composite (Lever 1.5/4/5 patches): targets F-Layer1 + F-Θsat. |
| `P10c` | `--v-phi-kind structural_competitive` + `--top-k 4` | 25–30 | Isolates P7 (Lever 3 row-softmax Φ̃): targets F1/F4. |
| `P10d` | P10c + all four P8 flags | 20–26 (actual: **32.99 final / 32.13 best**) | Full stack at vphi=16. Channel diagnostic confirms all four P8 patches landed their predicted signatures, but PPL ties P10a → V_φ is **capacity-bottlenecked** at vphi16, not channel-shape-bottlenecked. |
| `P10e` | P10d + `v_phi_phi_hidden=64, v_phi_theta_hidden=64` | 24–28 (actual: **31.89 final / 31.12 best**) | V_φ capacity ablation. R(ℓ) lifted 0.04 → 0.139 (close to predicted 0.15–0.30), PPL improved only ~1 PPL → V_φ width is *not* the binding constraint. **Asymptote is set by V_θ.** |
| `P10f` | P10e + `v_hidden=2048` (V_θ MLP doubled) | **26–29** | **V_θ-ceiling test.** V_θ params 2.3M → 8.9M (4× growth, total 16M → 22M). Pre-registered: PPL drops 31.9 → 26–29, mean R(ℓ) drops 0.139 → 0.05–0.10, |Θ| stays ~0.15–0.25, s_ℓ may flatten. Wall ~110–130 min on H100. |
| `P10g` | P10f + **16 000 steps** (2× training budget) | **26–28** (actual: **26.42 best / 27.16 final**) | **Training-budget disambiguator.** P10f landed at 28.67 (partial confirmation bracket). This doubles the step budget to test whether the ~28.5 PPL is a training-budget ceiling or a true corpus-information ceiling. Same architecture/LR/corpus. Result: PPL drops to 26.42 (best @ step 14400) → confirms corpus-information ceiling, not architecture ceiling. Decision: pivot to corpus scale-up. |
| `P10h` | P10g architecture + **20M tokens** (4× corpus) | **20–24** (actual: **26.43 best / 27.07 final**) | **Corpus scale-up.** P10g confirmed the 5M-token corpus is the binding constraint (PPL 26.42, slight overfit at 16k steps). This quadruples the training corpus to 20M GPT-2 BPE tokens while keeping P10f/g architecture (v_hidden=2048, full P5+P7+P8 stack) and 16k steps. Result: PPL 26.43 (best @ step 14800) — identical to P10g despite 4× data. Train-val gap = 0.027 nats (zero overfit). **Conclusion: the architecture is now the binding constraint.** The 22M-param PARFLM with v_hidden=2048 has reached its representational ceiling at ~26.4 PPL regardless of corpus size. Further progress requires architectural augmentation (FockPARFLM, deeper V_θ) or the full EOM simulator programme. |

## Decision rule (post-P10f)

P10e landed at 31.89 PPL with mean R(ℓ) = 0.139 — V_φ channels healthy, V_φ contribution lifted 3.6×, yet PPL barely moved. This is the **"V_θ-ceiling"** signature: V_θ does ~86% of the dynamics on average and sets the asymptote. **The next decision rests on P10f** (V_θ width doubled to 2048):

* `val_ppl(P10f) ≤ 28` ⇒ **V_θ ceiling confirmed.** V_θ width × depth scaling becomes the load-bearing lever for PARFLM. Schedule P10g at v_hidden=4096 (or v_depth=4) + paired-seed n=3 confirmation of P10f.
* `28 < val_ppl(P10f) ≤ 30` ⇒ **partial confirmation.** V_θ helps but TinyStories has a corpus-information ceiling around 27–28 PPL. Pivot to corpus scale (`max_train_tokens=20M`) or longer training (12k–16k steps) before further architectural width.
* `val_ppl(P10f) > 30` AND mean R(ℓ) drops ≥ 50% ⇒ **V_θ is also under-utilised but doesn't move the asymptote.** Pivot to corpus scale; the model is *information-bounded*, not *parameter-bounded*.
* `val_ppl(P10f) > 30` AND mean R(ℓ) unchanged ⇒ **V_θ scaling failed to engage** (e.g. optimisation issue with bigger V_θ MLP at the same LR). Re-tune learning rate (try 3e-4 instead of 5e-4) and re-run P10f.

References: `docs/PARF_Augmented_SPLM_Architecture_v2.md` §10 + §10.9 (P8); `docs/PARF-SPLM_Path_Forward_and_Experiments.md` §9.5 + §9.6 (P10) + §9.6 P10e/P10f pre-registrations.

## 0. Environment setup + cell selector

Pick which P10 cell to run via the `CELL` constant. Other knobs are fixed to the v3 paper's TinyStories scale-up cell shape so the comparison to SPLM em_ln 33.55 is apples-to-apples.

**Colab bootstrap (automatic).** When run on Google Colab the next cell:

1. mounts your Google Drive at `/content/drive`,
2. shallow-clones (`--depth 1 --branch main`) the public `dimitarpg13/semsimula` repo into `/content/semsimula` (or refreshes it if already present),
3. symlinks the data cache (`notebooks/conservative_arch/data`) to `/content/drive/MyDrive/semsimula_parflm/data` so the TinyStories tokenisation only happens once, ever,
4. routes **all** training outputs (ckpt, training log, val PPL plot, P6 diagnostic, `s_ℓ` profile) to `/content/drive/MyDrive/semsimula_parflm/p10_tinystories/{cell}/seed{SEED}/` — i.e. nothing important is written into the ephemeral `/content/semsimula` clone.

When run locally (off Colab) the notebook walks up from the CWD to find the repo root and writes to the in-repo path `parf/results/p10_tinystories/{cell}/seed{SEED}/` as before.

In [ ]:
# ===== Pick the cell to run =====
CELL = 'P10h'   # one of: 'P10a' | 'P10b' | 'P10c' | 'P10d' | 'P10e' | 'P10f' | 'P10g' | 'P10h'
SEED = 0

# ===== Source-of-truth repo + GDrive output dir =====
REPO_URL          = 'https://github.com/dimitarpg13/semsimula-paper.git'
REPO_BRANCH       = 'main'
COLAB_REPO_PATH   = '/content/semsimula-paper'                     # ephemeral source
GDRIVE_OUT_REL    = 'semsimula_parflm'                        # under /content/drive/MyDrive/

import os, sys, shutil, subprocess
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules
print(f'IN_COLAB = {IN_COLAB}')


def _sh(cmd: str) -> None:
    """Run a shell command, stream output, raise on non-zero exit."""
    print(f'$ {cmd}')
    r = subprocess.run(cmd, shell=True)
    if r.returncode != 0:
        raise RuntimeError(f'command failed (exit {r.returncode}): {cmd}')


if IN_COLAB:
    # ---------- (a) Mount Google Drive ----------
    from google.colab import drive  # type: ignore
    drive.mount('/content/drive', force_remount=False)
    GDRIVE_OUT = Path('/content/drive/MyDrive') / GDRIVE_OUT_REL
    GDRIVE_OUT.mkdir(parents=True, exist_ok=True)
    print(f'GDrive output root = {GDRIVE_OUT}')

    # ---------- (b) Shallow-clone (or refresh) the semsimula repo ----------
    REPO_ROOT = Path(COLAB_REPO_PATH)
    if not (REPO_ROOT / '.git').exists():
        if REPO_ROOT.exists():
            shutil.rmtree(REPO_ROOT)
        _sh(
            f'git clone --depth 1 --branch {REPO_BRANCH} '
            f'{REPO_URL} {REPO_ROOT}'
        )
    else:
        try:
            _sh(f'git -C {REPO_ROOT} fetch --depth 1 origin {REPO_BRANCH}')
            _sh(f'git -C {REPO_ROOT} reset --hard origin/{REPO_BRANCH}')
        except RuntimeError as e:
            print(f'WARNING: could not refresh repo ({e}); using existing checkout.')

    # ---------- (c) Pin tokenisation cache to Drive (persists across sessions) ----------
    DATA_CACHE = GDRIVE_OUT / 'data'
    DATA_CACHE.mkdir(exist_ok=True)
    repo_data_dir = REPO_ROOT / 'notebooks' / 'conservative_arch' / 'data'
    # Shallow clone may or may not include this stub; replace it with a Drive symlink.
    if repo_data_dir.is_symlink():
        repo_data_dir.unlink()
    elif repo_data_dir.exists():
        try:
            repo_data_dir.rmdir()           # only succeeds if empty
        except OSError:
            print(f'NOTE: {repo_data_dir} is non-empty; leaving as-is (no Drive symlink).')
    if not repo_data_dir.exists():
        repo_data_dir.symlink_to(DATA_CACHE, target_is_directory=True)
        print(f'data cache symlink: {repo_data_dir} -> {DATA_CACHE}')

    # ---------- (d) Output root: ckpt / log / plots / diagnostic all live on Drive ----------
    RESULTS_ROOT = GDRIVE_OUT / 'p10_tinystories'
    RESULTS_ROOT.mkdir(parents=True, exist_ok=True)

    # ---------- (e) Make sure the data_module's deps are present ----------
    _sh('pip install -q transformers huggingface_hub pyarrow')

else:
    # ---------- Local / non-Colab: locate repo root by walking up from CWD ----------
    REPO_ROOT = Path.cwd()
    while REPO_ROOT != REPO_ROOT.parent and not (REPO_ROOT / 'notebooks').exists():
        REPO_ROOT = REPO_ROOT.parent
    if not (REPO_ROOT / 'notebooks').exists():
        raise RuntimeError(
            'Could not locate the semsimula repo root from the notebook CWD. '
            'cd into the repo before launching the notebook.'
        )
    RESULTS_ROOT = (
        REPO_ROOT / 'notebooks' / 'conservative_arch' / 'parf'
        / 'results' / 'p10_tinystories'
    )
    RESULTS_ROOT.mkdir(parents=True, exist_ok=True)

# ===== sys.path so we can import data_module / model_parf / scaleup utils =====
PARF_DIR    = REPO_ROOT / 'notebooks' / 'conservative_arch' / 'parf'
SCALEUP_DIR = REPO_ROOT / 'notebooks' / 'conservative_arch' / 'scaleup'
DATA_DIR    = REPO_ROOT / 'notebooks' / 'conservative_arch'
for p in (str(REPO_ROOT), str(DATA_DIR), str(SCALEUP_DIR), str(PARF_DIR)):
    if p not in sys.path:
        sys.path.insert(0, p)

print(f'\nREPO_ROOT     = {REPO_ROOT}')
print(f'PARF_DIR      = {PARF_DIR}')
print(f'SCALEUP_DIR   = {SCALEUP_DIR}')
print(f'RESULTS_ROOT  = {RESULTS_ROOT}')
print(f'CELL = {CELL!r}  SEED = {SEED}')

## 0.5. One-shot: migrate the pilot P10a run from a Drive `_inbox`

The original pilot P10a run (val PPL **32.60**, the run that broke the SPLM em_ln 33.55 wall) was launched before this notebook had GDrive routing, so its artefacts live on your local Mac at `~/Downloads/semsimula_pilot.2/results/` instead of on Drive. This cell, run **once**, lifts those files into `RESULTS_ROOT/P10a/seed0/` so the dashboard at the bottom (§9) renders the P10a row alongside the new P10b/c/d/e runs.

**To use it:**

1. On your **Mac**, with Google Drive Desktop installed, copy the four pilot files into the Drive-synced staging folder:

   ```bash
   GDRIVE_BASE="$HOME/Library/CloudStorage/GoogleDrive-<your-email>/My Drive"
   # (or wherever your Drive Desktop sync mounts; common alternative: $HOME/Google Drive)
   mkdir -p "$GDRIVE_BASE/semsimula_parflm/_inbox"
   cp -v ~/Downloads/semsimula_pilot.2/results/parf_structural_vphi16_sparse_k4_scaleup_scaleup_seed0_* \
        "$GDRIVE_BASE/semsimula_parflm/_inbox/"
   # If you also want the existing P6 diagnostic surfaced under P10a/seed0/p6_diagnostic/:
   cp -rv notebooks/conservative_arch/parf/diagnostics/results/tinystories_p10a_pilot_seed0_k4 \
        "$GDRIVE_BASE/semsimula_parflm/_inbox/p6_diagnostic"
   ```

2. Wait ~30 s for Google Drive Desktop to upload the files (they'll appear in the Drive web UI under `My Drive/semsimula_parflm/_inbox/`).

3. **On Colab**, run the next cell. It detects files in `_inbox`, moves each one into the canonical `RESULTS_ROOT/P10a/seed0/` layout, and empties the inbox. Idempotent (re-running on an empty inbox is a no-op).

In [ ]:
# ===== One-shot: migrate any files in /content/drive/MyDrive/semsimula_parflm/_inbox/ =====
# into RESULTS_ROOT/P10a/seed0/ .  Idempotent (no-op if _inbox is missing or empty).

if IN_COLAB:
    INBOX = Path('/content/drive/MyDrive') / GDRIVE_OUT_REL / '_inbox'
    if not INBOX.exists():
        print(f'(no _inbox at {INBOX}; nothing to migrate — skipping)')
    else:
        items = sorted(INBOX.iterdir())
        if not items:
            print(f'(_inbox at {INBOX} is empty; nothing to migrate)')
        else:
            DEST = RESULTS_ROOT / 'P10a' / f'seed{SEED}'
            DEST.mkdir(parents=True, exist_ok=True)
            print(f'Migrating {len(items)} item(s) from {INBOX} → {DEST}')
            moved = []
            for src in items:
                dst = DEST / src.name
                if src.is_file():
                    if dst.exists():
                        dst.unlink()
                    shutil.move(str(src), str(dst))
                    moved.append(f'  • {src.name}  ({dst.stat().st_size / 1e6:.1f} MB)')
                elif src.is_dir():
                    if dst.exists():
                        shutil.rmtree(dst)
                    shutil.move(str(src), str(dst))
                    moved.append(f'  • {src.name}/  (dir)')
            print('\n'.join(moved))
            # Clean up the now-empty inbox.
            try:
                INBOX.rmdir()
                print(f'\nRemoved empty _inbox at {INBOX}.')
            except OSError:
                print(f'\n(_inbox at {INBOX} not empty after move; leaving it.)')
else:
    print('(migration cell is Colab-only; skipping in local mode)')

## 1. Disable TF32, set seeds, pick device

In [ ]:
import torch
import numpy as np

# ----- TF32 OFF on Ampere/Hopper -----
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False
torch.set_float32_matmul_precision('highest')

torch.manual_seed(SEED)
np.random.seed(SEED)
rng = np.random.default_rng(SEED)

if torch.cuda.is_available():
    device = 'cuda'
    torch.cuda.manual_seed_all(SEED)
    cap = torch.cuda.get_device_capability()
    print(f'CUDA: {torch.cuda.get_device_name(0)}  sm_{cap[0]}{cap[1]}  '
          f'mem={torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
    if cap[0] < 8:
        print(f'  WARNING: compute capability < 8.0 ({cap}); P10 cell was '
              f'designed for A100 (sm_80) or H100 (sm_90).')
    print(f'  TF32 matmul = {torch.backends.cuda.matmul.allow_tf32}  (must be False)')
    print(f'  TF32 cuDNN  = {torch.backends.cudnn.allow_tf32}  (must be False)')
elif torch.backends.mps.is_available():
    device = 'mps'
    print('MPS device — TF32 toggles are CUDA-only.')
else:
    device = 'cpu'
    print('CPU only — P10 cells take ~days on CPU; smoke-test mode strongly recommended.')
print(f'\ndevice = {device}')

## 2. Load TinyStories + the bundled logfreq surprisal

In [ ]:
from data_module import load_tiny_stories, get_batch

# P10h scales up the corpus from 5M → 20M tokens; all prior cells use 5M.
# TinyStories has 4 train shards (~5M tokens each); P10h loads all 4.
if CELL == 'P10h':
    MAX_TRAIN_TOKENS = 20_000_000
    N_TRAIN_FILES = 4
else:
    MAX_TRAIN_TOKENS = 5_000_000
    N_TRAIN_FILES = 1
train_ids, val_ids = load_tiny_stories(
    n_train_files=N_TRAIN_FILES, max_train_tokens=MAX_TRAIN_TOKENS
)
print(f'tokens: train={len(train_ids):,}  val={len(val_ids):,}  '
      f'(train capped at {MAX_TRAIN_TOKENS:,})')
print(f'train_ids dtype={train_ids.dtype}  range=[{train_ids.min()}, {train_ids.max()}]')

BUNDLED_LOGFREQ = SCALEUP_DIR / 'results' / 'logfreq_surprisal_tinystories.npy'
DRIVE_LOGFREQ   = RESULTS_ROOT / 'logfreq_surprisal_tinystories.npy'
if BUNDLED_LOGFREQ.exists():
    LOGFREQ_PATH = BUNDLED_LOGFREQ
    print(f'Using bundled TinyStories logfreq surprisal: {LOGFREQ_PATH}')
elif DRIVE_LOGFREQ.exists():
    # Built on a previous Colab session; persists on GDrive for re-use.
    LOGFREQ_PATH = DRIVE_LOGFREQ
    print(f'Using Drive-cached TinyStories logfreq surprisal: {LOGFREQ_PATH}')
else:
    # On-the-fly fallback (Colab / fresh checkout): build from train_ids.
    VOCAB_SIZE = 50257
    counts = np.bincount(train_ids.astype(np.int64), minlength=VOCAB_SIZE).astype(np.float64)
    p = (counts + 1.0) / (counts.sum() + VOCAB_SIZE)
    surprisal = (-np.log(p)).astype(np.float32)
    LOGFREQ_PATH = DRIVE_LOGFREQ
    LOGFREQ_PATH.parent.mkdir(parents=True, exist_ok=True)
    np.save(LOGFREQ_PATH, surprisal)
    print(f'Built logfreq from train_ids; saved to {LOGFREQ_PATH}')

## 3. Translate `CELL` switch into trainer flags + build config

All four P10 cells share the same scale-up cell shape (`mode='scaleup'`: d=256, L=8, T=512, B=16, v_hidden=1024, 8000 steps). They differ only in which V_φ flags are enabled.

In [ ]:
from train_parf_scaleup import build_config
from model_parf import PARFLM
from model_parf_sparse import SparsePARFLM, SparsePARFConfig
import torch.nn.functional as F

P10_RECIPES = {
    'P10a': dict(
        v_phi_kind='structural',
        ln_before_distance=False, per_layer_v_phi_scale=False,
        theta_activation='tanh', theta_form='mlp',
        v_phi_phi_hidden=None, v_phi_theta_hidden=None,    # → scaleup default 16
    ),
    'P10b': dict(
        v_phi_kind='structural',
        ln_before_distance=True, per_layer_v_phi_scale=True,
        theta_activation='softsign', theta_form='bilinear',
        v_phi_phi_hidden=None, v_phi_theta_hidden=None,    # → scaleup default 16
    ),
    'P10c': dict(
        v_phi_kind='structural_competitive',
        ln_before_distance=False, per_layer_v_phi_scale=False,
        theta_activation='tanh', theta_form='mlp',
        v_phi_phi_hidden=None, v_phi_theta_hidden=None,    # → scaleup default 16
    ),
    'P10d': dict(
        v_phi_kind='structural_competitive',
        ln_before_distance=True, per_layer_v_phi_scale=True,
        theta_activation='softsign', theta_form='bilinear',
        v_phi_phi_hidden=None, v_phi_theta_hidden=None,    # → scaleup default 16
    ),
    # ---- P10e: V_φ capacity ablation ----
    # Same composition as P10d (P5 + P7 + P8 full stack), but quadruples the
    # V_φ inner widths from 16 → 64.  Falsification test for the
    # "V_φ at vphi16 is capacity-bottlenecked" hypothesis from the s_ℓ + R(ℓ)
    # diagnostic of the P10d run.  Pre-registered predictions:
    #   • s_ℓ plateau ceiling rises 0.14 → 0.4–0.7
    #   • mean R(ℓ) rises 0.04 → 0.15–0.30
    #   • val PPL drops 32–33 → 24–28 (possibly into the 20s).
    # ACTUAL OUTCOME: best 31.12 / final 31.89, mean R(ℓ) = 0.139 (just under the
    # predicted band).  R(ℓ) prediction confirmed; PPL prediction falsified.
    # → V_φ width is NOT the binding constraint.  V_θ ceiling dominates.
    'P10e': dict(
        v_phi_kind='structural_competitive',
        ln_before_distance=True, per_layer_v_phi_scale=True,
        theta_activation='softsign', theta_form='bilinear',
        v_phi_phi_hidden=64, v_phi_theta_hidden=64,
        v_hidden=None, v_depth=None,                       # → scaleup default 1024 / 3
    ),
    # ---- P10f: V_θ ceiling test ----
    # P10e + double V_θ MLP width 1024 → 2048.  V_θ params 2.3M → 8.9M (4×
    # because both inner matmuls grow 4×).  Total model 16M → 22M params.
    # Tests the V_θ-ceiling hypothesis after P10e showed V_φ is healthy
    # (mean R(ℓ) = 0.139 = "perturbation-but-non-trivial regime") yet PPL
    # barely moved → asymptote is set by V_θ, not V_φ.
    # Pre-registered predictions:
    #   • val PPL drops 31.9 → 26–29 (binary test for V_θ ceiling)
    #   • mean R(ℓ) DROPS 0.139 → 0.05–0.10 (V_θ force grows; V_φ_force constant)
    #   • |Θ| stays bounded ~0.15–0.25 (Patches C+D continue to work)
    #   • s_ℓ profile may flatten (less reliance on per-layer V_φ scaling)
    # Wall-clock: V_θ is ~half the total compute; expect 1.5-1.8× P10e's
    # wall-clock on H100 (~110-130 min for 8000 steps).
    'P10f': dict(
        v_phi_kind='structural_competitive',
        ln_before_distance=True, per_layer_v_phi_scale=True,
        theta_activation='softsign', theta_form='bilinear',
        v_phi_phi_hidden=64, v_phi_theta_hidden=64,
        v_hidden=2048, v_depth=None,                       # ← the new dim
    ),
    # ---- P10g: Training-budget disambiguator ----
    # Same architecture as P10f (v_hidden=2048, full P5+P7+P8 stack) but
    # 16000 steps (2× P10f's budget).  P10f landed at best val PPL = 28.67
    # (partial confirmation bracket: 28 < PPL ≤ 30).  This run tests whether
    # the ~28.5 asymptote is a training-budget ceiling or a true
    # corpus-information ceiling on TinyStories at 5M tokens.
    # Pre-registered predictions:
    #   • val PPL drops 28.67 → 26–28 if training-budget-limited
    #   • val PPL plateaus at ~28–29 if corpus-information-limited
    # Decision rule:
    #   • PPL < 27.5 → training was the bottleneck; schedule v_hidden=4096
    #   • PPL ≥ 28   → corpus ceiling confirmed; pivot to 20M tokens
    # Wall-clock: ~2× P10f = 220–260 min on H100.
    'P10g': dict(
        v_phi_kind='structural_competitive',
        ln_before_distance=True, per_layer_v_phi_scale=True,
        theta_activation='softsign', theta_form='bilinear',
        v_phi_phi_hidden=64, v_phi_theta_hidden=64,
        v_hidden=2048, v_depth=None,
        _steps_override=16000,                             # 2× default
    ),
    # ---- P10h: Corpus scale-up ----
    # P10g confirmed the corpus-information ceiling (best PPL 26.42 at 5M tokens,
    # slight overfit visible at 16k steps).  This experiment quadruples the
    # training corpus to 20M GPT-2 BPE tokens (MAX_TRAIN_TOKENS override handled
    # in the data-loading cell above) while keeping the P10f/g architecture
    # (v_hidden=2048, full P5+P7+P8 stack) and 16k steps.
    # Pre-registered predictions:
    #   • val PPL drops 26.42 → 20–24 (information ceiling lifts with 4× data)
    #   • No overfit (train PPL ≈ val PPL throughout)
    #   • mean R(ℓ) may rise slightly (V_φ has more signal to work with)
    # Decision rule:
    #   • PPL ≤ 20 → success criterion met; publish result + plan v_hidden=4096
    #   • 20 < PPL ≤ 24 → on track; consider 40M tokens or longer training
    #   • PPL > 24 → architecture may need deeper V_θ (v_depth=4); diagnose
    # Wall-clock: ~220–260 min on H100 (same as P10g; more data per step but
    # same step count).
    'P10h': dict(
        v_phi_kind='structural_competitive',
        ln_before_distance=True, per_layer_v_phi_scale=True,
        theta_activation='softsign', theta_form='bilinear',
        v_phi_phi_hidden=64, v_phi_theta_hidden=64,
        v_hidden=2048, v_depth=None,
        _steps_override=16000,                             # same budget as P10g
    ),
}
if CELL not in P10_RECIPES:
    raise ValueError(f'CELL must be one of {list(P10_RECIPES)}; got {CELL!r}')
recipe = P10_RECIPES[CELL]

# Extract notebook-level overrides (prefixed with _) before passing to build_config.
_STEPS_OVERRIDE = recipe.pop('_steps_override', None)

print(f'P10 recipe for {CELL}:')
for k, v in recipe.items():
    print(f'  {k:28s} = {v!r}')
if _STEPS_OVERRIDE is not None:
    print(f'  {"_steps_override":28s} = {_STEPS_OVERRIDE}')

# Knobs that build_config accepts as named kwargs (None → mode default):
_BC_OVERRIDES = ('v_phi_phi_hidden', 'v_phi_theta_hidden', 'v_hidden', 'v_depth')
cfg, train_cfg, tag = build_config(
    mode='scaleup',
    logfreq_path=str(LOGFREQ_PATH),
    fixed_gamma=None,
    sparse_top_k=4,                      # P5 sparsity is on for all four
    sparse_score_head_hidden=32,
    sparse_gumbel_tau_init=1.0,
    sparse_gumbel_tau_min=0.1,
    sparse_gumbel_noise=True,
    **{k: recipe.get(k) for k in _BC_OVERRIDES},
    **{k: v for k, v in recipe.items() if k not in _BC_OVERRIDES},
)

# Apply per-cell training overrides (e.g. P10g doubles the step budget).
if _STEPS_OVERRIDE is not None:
    train_cfg['steps'] = _STEPS_OVERRIDE
    train_cfg['warmup_steps'] = min(train_cfg['warmup_steps'],
                                    _STEPS_OVERRIDE // 20)
full_tag = f'{tag}_seed{SEED}'
is_sparse = isinstance(cfg, SparsePARFConfig)
print(f'\ntag = {full_tag}')
print(f'cell shape: d={cfg.d}, L={cfg.L}, T={cfg.max_len}, '
      f'v_hidden={cfg.v_hidden}, v_phi_phi_hidden={cfg.v_phi_phi_hidden}, '
      f'v_phi_theta_hidden={cfg.v_phi_theta_hidden}')
print(f'train_cfg = {train_cfg}')
print(f'sparse: {is_sparse}, top_k={getattr(cfg, "top_k", None)}')

In [ ]:
torch.manual_seed(SEED)
Model = SparsePARFLM if is_sparse else PARFLM
model = Model(cfg).to(device)
n_total = sum(p.numel() for p in model.parameters())
n_v_theta = sum(p.numel() for p in model.V_theta.parameters())
n_v_phi = sum(p.numel() for p in model.V_phi.parameters())
n_pls = model.raw_v_phi_scale.numel() if model.raw_v_phi_scale is not None else 0
n_score = (sum(p.numel() for p in model.score_head.parameters())
           if is_sparse else 0)
print(f'params: total={n_total:,}  V_theta={n_v_theta:,}  '
      f'V_phi={n_v_phi:,}  per_layer_scale={n_pls}  score_head={n_score:,}')
if model.raw_v_phi_scale is not None:
    print(f'init s_ℓ = {F.softplus(model.raw_v_phi_scale).detach().tolist()}')

## 4. Causal-violation probe (gates training)

In [ ]:
from causal_probe_parf import assert_causal
assert_causal(model, vocab_size=cfg.vocab_size, T=32, seed=SEED)
print('causal probe OK — no future-position leak.')

## 5. Train

8000 steps, batch=16, T=512, AdamW(0.9, 0.95), cosine LR with 400-step warmup. Cell-by-cell wall-clock estimate: A100-40GB ~16-20 h, H100-80GB ~6-9 h. Plan to leave the kernel running overnight.

If you want to test the loop end-to-end first without burning the full budget, override `STEPS = 200, EVAL_INTERVAL = 100` in the cell below before running.

In [ ]:
import math, time, json

BATCH = train_cfg['batch_size']
BLOCK = train_cfg['block_size']
STEPS = train_cfg['steps']
LR = train_cfg['lr']
WD = train_cfg['weight_decay']
WARMUP = train_cfg['warmup_steps']
GRAD_CLIP = train_cfg['grad_clip']
EVAL_INTERVAL = train_cfg['eval_interval']
EVAL_ITERS = train_cfg['eval_iters']
LOG_INTERVAL = train_cfg['log_interval']

# Sparse-PARF Gumbel-tau anneal (no-op if dense).
from train_parf_scaleup import tau_schedule
GUMBEL_ANNEAL_FRAC = 0.8

def lr_at(step):
    if step < WARMUP:
        return LR * (step + 1) / WARMUP
    progress = (step - WARMUP) / max(STEPS - WARMUP, 1)
    return LR * 0.5 * (1.0 + math.cos(math.pi * min(progress, 1.0)))

@torch.no_grad()
def evaluate():
    model.eval()
    losses = []
    for _ in range(EVAL_ITERS):
        xb, yb = get_batch(val_ids, BATCH, BLOCK, rng)
        x = torch.from_numpy(xb).to(device)
        y = torch.from_numpy(yb).to(device)
        with torch.enable_grad():
            _, loss = model(x, y)
        losses.append(loss.item())
    model.train()
    return float(np.mean(losses))

opt = torch.optim.AdamW(
    model.parameters(), lr=LR, betas=(0.9, 0.95), weight_decay=WD,
)
model.train()

log = []
t0 = time.time()
for step in range(STEPS):
    for g in opt.param_groups:
        g['lr'] = lr_at(step)
    if is_sparse:
        model.set_gumbel_tau(tau_schedule(step, cfg.gumbel_tau_init,
                                          cfg.gumbel_tau_min, STEPS,
                                          GUMBEL_ANNEAL_FRAC))
    xb, yb = get_batch(train_ids, BATCH, BLOCK, rng)
    x = torch.from_numpy(xb).to(device)
    y = torch.from_numpy(yb).to(device)
    _, loss = model(x, y)
    opt.zero_grad(set_to_none=True)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
    opt.step()

    if (step + 1) % LOG_INTERVAL == 0 or step == 0:
        msg = (f'[{CELL}] step {step + 1:>5}/{STEPS}  '
               f'lr={lr_at(step):.2e}  '
               f'train_loss={loss.item():.4f}  '
               f'wall={time.time() - t0:.0f}s')
        print(msg)
    if (step + 1) % EVAL_INTERVAL == 0 or (step + 1) == STEPS:
        val_loss = evaluate()
        val_ppl = math.exp(val_loss)
        print(f'  >> val_loss={val_loss:.4f}  val_ppl={val_ppl:.2f}')
        log.append({'step': step + 1, 'val_loss': val_loss,
                    'val_ppl': val_ppl, 'train_loss': loss.item()})

print(f'\n[{CELL}] Training done.  total wall = {time.time() - t0:.0f}s  '
      f'final val_ppl = {log[-1]["val_ppl"]:.2f}')

## 6. Save checkpoint and training log

In [ ]:
RUN_DIR = RESULTS_ROOT / CELL / f'seed{SEED}'
RUN_DIR.mkdir(parents=True, exist_ok=True)
ckpt_path = RUN_DIR / f'{full_tag}_ckpt_latest.pt'
log_path = RUN_DIR / f'{full_tag}_training_log.jsonl'

import dataclasses
torch.save(
    {
        'model_state_dict': model.state_dict(),
        'model_cfg': dataclasses.asdict(cfg),
        'variant': full_tag,
        'cell': CELL,
        'step': STEPS,
        'final_val_ppl': log[-1]['val_ppl'],
    },
    ckpt_path,
)
with open(log_path, 'w') as f:
    for row in log:
        f.write(json.dumps(row) + '\n')
print(f'wrote ckpt: {ckpt_path}')
print(f'wrote log : {log_path}')

## 7. Plot val PPL trajectory + bookend baselines

Two horizontal reference lines: SPLM em_ln 33.55 (the "wall") and the 20 PPL target. Anything below 33.55 is non-trivial; anything below 20 is a P10d win.

In [ ]:
import matplotlib.pyplot as plt

steps_log = [r['step'] for r in log]
ppl_log = [r['val_ppl'] for r in log]

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(steps_log, ppl_log, marker='o', color='#3a6ea5', label=f'{CELL}')
ax.axhline(33.55, color='#888', linestyle='--', label='SPLM em_ln (33.55, the wall)')
ax.axhline(20.00, color='#1a8c1a', linestyle='--', label='P10d target (≤ 20)')
ax.axhline( 7.81, color='#b03030', linestyle=':',  label='MatchedGPT (7.81, upper bound)')
ax.set_xlabel('train step')
ax.set_ylabel('val PPL (log scale)')
ax.set_yscale('log')
ax.set_title(f'{CELL} on TinyStories — val PPL trajectory  (final = {ppl_log[-1]:.2f})')
ax.legend(loc='upper right')
ax.grid(True, which='both', alpha=0.3)
plt.tight_layout()
plt.savefig(RUN_DIR / f'{full_tag}_val_ppl.png', dpi=120)
plt.show()

## 8. (Optional) Run the V_φ channel diagnostic on the trained P10 ckpt

Verifies the F-Layer1 / F-Θsat predictions for P10b/d (where Patch A + Patch B should equalise R(ℓ) and de-saturate Θ in deep layers). Skips automatically for P10a/c if you don't want to wait.

In [ ]:
RUN_DIAGNOSTIC = True   # set False to skip
if RUN_DIAGNOSTIC:
    import subprocess
    DIAG_OUT = RUN_DIR / 'p6_diagnostic'
    DIAG_OUT.mkdir(parents=True, exist_ok=True)
    diag_script = PARF_DIR / 'diagnostics' / 'diagnose_v_phi_channels.py'
    result = subprocess.run(
        [
            sys.executable, str(diag_script),
            '--ckpt', str(ckpt_path),
            '--out', str(DIAG_OUT),
            '--n-batches', '4',
            '--batch-size', '8',
            '--block-size', '128',     # T=128 is enough for the channel hist; faster than full T=512
            '--device', device,
            '--seed', str(SEED),
        ],
        capture_output=True, text=True,
    )
    print(result.stdout[-3000:])
    if result.returncode != 0:
        print('STDERR:\n' + result.stderr[-2000:])
        raise RuntimeError(f'diagnostic exited with code {result.returncode}')

    from IPython.display import Image, display, Markdown
    for fname in ('channels.png', 'gradient_ratio.png'):
        p = DIAG_OUT / fname
        if p.exists():
            display(Markdown(f'### {fname}'))
            display(Image(filename=str(p)))
    summary_md = DIAG_OUT / 'summary.md'
    if summary_md.exists():
        display(Markdown('### diagnostic summary'))
        display(Markdown(summary_md.read_text()))
else:
    print('Skipping V_φ diagnostic.  Set RUN_DIAGNOSTIC = True above to enable.')

In [ ]:
# Per-layer s_ℓ profile (only meaningful for P10b/d which have --per-layer-v-phi-scale).
if model.raw_v_phi_scale is not None:
    s_ell = F.softplus(model.raw_v_phi_scale).detach().cpu().numpy()
    plt.figure(figsize=(7, 3.5))
    plt.bar(np.arange(1, len(s_ell) + 1), s_ell, color='#3a6ea5')
    plt.axhline(F.softplus(torch.tensor(-3.0)).item(), color='gray',
                linestyle='--', alpha=0.6, label='init s_ℓ ≈ 0.0486')
    plt.xlabel('layer ℓ')
    plt.ylabel('learned s_ℓ = softplus(σ_ℓ)')
    plt.title(f'{CELL} per-layer V_φ scale after {STEPS} steps')
    plt.legend()
    plt.tight_layout()
    plt.savefig(RUN_DIR / 'per_layer_scale.png', dpi=120)
    plt.show()
    print(f's_ℓ profile = {s_ell.tolist()}')
else:
    print(f'(per_layer_v_phi_scale was OFF for {CELL}.)')

## 9. P10 ladder dashboard

Re-run this notebook once per cell (P10a → P10d), and the dashboard below auto-collects the final val PPL of every cell present under `RESULTS_ROOT/{cell}/seed{SEED}/`. On Colab that is `/content/drive/MyDrive/semsimula_parflm/p10_tinystories/{cell}/seed{SEED}/`, so each pass persists across kernel restarts and the table is cumulative.

In [ ]:
P10_DIR = RESULTS_ROOT
results = {}
for cell_name in ('P10a', 'P10b', 'P10c', 'P10d', 'P10e', 'P10f', 'P10g', 'P10h'):
    cell_dir = P10_DIR / cell_name / f'seed{SEED}'
    if not cell_dir.exists():
        results[cell_name] = None
        continue
    logs = sorted(cell_dir.glob('*_training_log.jsonl'))
    if not logs:
        results[cell_name] = None
        continue
    rows = [json.loads(line) for line in logs[-1].read_text().splitlines()]
    results[cell_name] = rows[-1]['val_ppl'] if rows else None

WALL = 33.55
TARGET = 20.00
MATCHED_GPT = 7.81
print(f'{"cell":<8} {"val_ppl":>10}    {"vs wall":>10}    {"vs target":>10}    verdict')
print('-' * 70)
for cell_name, ppl in results.items():
    if ppl is None:
        print(f'{cell_name:<8} {"—":>10}    {"—":>10}    {"—":>10}    (not run)')
        continue
    d_wall = ppl - WALL
    d_target = ppl - TARGET
    if ppl <= TARGET:
        verdict = '🎯 WIN'
    elif ppl <= 25:
        verdict = '↘ partial win'
    elif ppl <= WALL:
        verdict = '↘ beats wall'
    else:
        verdict = '↑ regression'
    print(f'{cell_name:<8} {ppl:>10.2f}    {d_wall:>+10.2f}    {d_target:>+10.2f}    {verdict}')
print(f'\nWall = SPLM em_ln {WALL}    Target = {TARGET}    MatchedGPT = {MATCHED_GPT}')

## 10. CLI equivalents (for SLURM / nohup launches)

```bash
# P10a — anchor (P5 sparse k=4, no P7, no P8)
python notebooks/conservative_arch/scaleup/train_parf_scaleup.py \
    --mode scaleup --seed 0 --top-k 4

# P10b — P5 + P8 composite
python notebooks/conservative_arch/scaleup/train_parf_scaleup.py \
    --mode scaleup --seed 0 --top-k 4 \
    --ln-before-distance --per-layer-v-phi-scale \
    --theta-activation softsign --theta-form bilinear

# P10c — P5 + P7 competitive Φ
python notebooks/conservative_arch/scaleup/train_parf_scaleup.py \
    --mode scaleup --seed 0 --top-k 4 \
    --v-phi-kind structural_competitive

# P10d — full P5 + P7 + P8 stack at vphi=16
python notebooks/conservative_arch/scaleup/train_parf_scaleup.py \
    --mode scaleup --seed 0 --top-k 4 \
    --v-phi-kind structural_competitive \
    --ln-before-distance --per-layer-v-phi-scale \
    --theta-activation softsign --theta-form bilinear

# P10e — V_phi capacity ablation: P10d + 4× wider V_phi
# Actual: best 31.12 / final 31.89 PPL; mean R(ℓ) lifted 0.04 → 0.139.
python notebooks/conservative_arch/scaleup/train_parf_scaleup.py \
    --mode scaleup --seed 0 --top-k 4 \
    --v-phi-kind structural_competitive \
    --ln-before-distance --per-layer-v-phi-scale \
    --theta-activation softsign --theta-form bilinear \
    --v-phi-phi-hidden 64 --v-phi-theta-hidden 64

# P10f — V_theta ceiling test: P10e + double V_theta MLP width (the 26-29 PPL target)
python notebooks/conservative_arch/scaleup/train_parf_scaleup.py \
    --mode scaleup --seed 0 --top-k 4 \
    --v-phi-kind structural_competitive \
    --ln-before-distance --per-layer-v-phi-scale \
    --theta-activation softsign --theta-form bilinear \
    --v-phi-phi-hidden 64 --v-phi-theta-hidden 64 \
    --v-hidden 2048

# P10g — Training-budget disambiguator: P10f architecture at 16k steps (2× budget)
# (no CLI flag for steps override; use the notebook with CELL='P10g', or
#  override STEPS=16000 manually when calling train_parf_scaleup.py)
python notebooks/conservative_arch/scaleup/train_parf_scaleup.py \
    --mode scaleup --seed 0 --top-k 4 \
    --v-phi-kind structural_competitive \
    --ln-before-distance --per-layer-v-phi-scale \
    --theta-activation softsign --theta-form bilinear \
    --v-phi-phi-hidden 64 --v-phi-theta-hidden 64 \
    --v-hidden 2048
# NOTE: manually set steps=16000 (the CLI uses 8000 by default for --mode scaleup)

# P10h — Corpus scale-up: P10g architecture + 20M tokens (4× corpus)
# Use the notebook with CELL='P10h' (handles MAX_TRAIN_TOKENS=20M automatically).
# For CLI launch, pass --max-train-tokens 20000000 (if supported) or modify the
# script's default.  Steps=16000 same as P10g.
python notebooks/conservative_arch/scaleup/train_parf_scaleup.py \
    --mode scaleup --seed 0 --top-k 4 \
    --v-phi-kind structural_competitive \
    --ln-before-distance --per-layer-v-phi-scale \
    --theta-activation softsign --theta-form bilinear \
    --v-phi-phi-hidden 64 --v-phi-theta-hidden 64 \
    --v-hidden 2048
# NOTE: manually set steps=16000, max_train_tokens=20_000_000
```

Output dirs (local CLI launches): `notebooks/conservative_arch/parf/results/p10_tinystories/{cell}/seed{seed}/`. The notebook above (when run on Colab) routes the same artefacts to `/content/drive/MyDrive/semsimula_parflm/p10_tinystories/{cell}/seed{seed}/` instead.